In [ ]:
import os
import re
from datetime import datetime

import ee
import geemap



#####################################
#    SET ALL PARAMETERS #############
#####################################

PROJECT_DIR = "projects/ee-tymc5571-cbi-module/assets/"
GEE_PROJECT = "ee-tymc5571-cbi-module"
CRS = "EPSG:4326" #this is the CRS that the data will be exported in

#ee.Authenticate()
ee.Authenticate(
    auth_mode='notebook',  # or 'gcloud' or 'appdefault' in other contexts
    scopes=['https://www.googleapis.com/auth/earthengine',
            'https://www.googleapis.com/auth/devstorage.read_write',
            'https://www.googleapis.com/auth/drive']
    #, quiet=False  # force re-authentication even if cached tokens exist
)
ee.Initialize(project=GEE_PROJECT)

# Set output region as desired
states = ee.FeatureCollection("TIGER/2018/States")

western_states = [
    'WA', 'OR', 'CA', 'ID', 'NV', 'MT', 'WY', 'UT',
    'CO', 'AZ', 'NM'#, 'ND', 'SD', 'NE', 'KS', 'OK', 'TX'
]
outputRegion = states.filter(
    ee.Filter.inList('STUSPS', western_states)
)

START_YEAR = 1984
END_YEAR = 2025
CRS = "EPSG:4326" #this is the CRS that the data will be exported in

# Data
lcms = ee.ImageCollection("USFS/GTAC/LCMS/v2024-10") 
lcpri = ee.ImageCollection("projects/sat-io/open-datasets/LCMAP/LCPRI")



In [14]:
# A few functions
def remove_to_bands_append2(image, og_ic):
    """
    Removes image ID prefixes (from toBands()) using IDs from the original ImageCollection.
    
    Args:
        image (ee.Image): The image with compound band names after toBands().
        og_ic (ee.ImageCollection): The original ImageCollection before stacking.
    
    Returns:
        ee.Image: Image with cleaned band names.
    """
    ids = ee.List(og_ic.aggregate_array('system:index'))

    def clean_name(name):
        name = ee.String(name)
        def iterate_fn(id_, acc):
            acc_str = ee.String(acc)
            id_str = ee.String(id_).cat('_')
            match = acc_str.slice(0, id_str.length()).equals(id_str)
            return ee.Algorithms.If(match, acc_str.slice(id_str.length()), acc_str)
        return ids.iterate(iterate_fn, name)

    old_names = image.bandNames()
    new_names = old_names.map(lambda b: clean_name(b))
    return image.rename(new_names)

def toBands_with_projection(collection):
    collection = ee.ImageCollection(collection)
    image = collection.toBands()
    reference_img = ee.Image(collection.first())
    return image.setDefaultProjection(reference_img.projection())

In [18]:
# /////////////////////////////////////////////////////////////////////////////////
# /////////////////////////////////////////////////////////////////////////////////
# ///////////       GENERATE ANNUAL CONSERVATIVE FOREST MASK      /////////////////
# /////////////////////////////////////////////////////////////////////////////////
# /////////////////////////////////////////////////////////////////////////////////

# Print image collections to view if necessary
# Uncomment the following lines to print the collections
# print('LCPRI:', lcpri.getInfo())
# print('LCMS:', lcms.getInfo())

# Reclassify an image to binary (1 = forest)
def reclassify_image_binary(value):
    def inner(image):
        binary = image.eq(value).selfMask()
        return binary.copyProperties(image, ['system:time_start'])
    return inner

# AND operation between two image collections
def combine_collections_with_and(collection1, collection2, and_name):
    list1 = collection1.toList(collection1.size())
    list2 = collection2.toList(collection2.size())
    zipped = list1.zip(list2)

    def and_func(pair):
        image1 = ee.Image(ee.List(pair).get(0))
        image2 = ee.Image(ee.List(pair).get(1))
        result = image1.And(image2).rename(and_name)
        return result.copyProperties(image1, ['system:time_start'])

    return ee.ImageCollection(zipped.map(and_func))

# Add year property from system:time_start
def add_year_property(image):
    year = image.date().get('year')
    return image.set('year', year)

# Filter LCPRI and LCMS collections by date
lcpri_cover = lcpri.filterDate(str(START_YEAR), str(END_YEAR))
lcms_cover = lcms.filter(ee.Filter.eq('study_area', 'CONUS')) \
                 .filterDate(str(START_YEAR), str(END_YEAR)) \
                 .select('Land_Cover')

# Reclassify forest (LCPRI=4, LCMS=1)
lcpri_for = lcpri_cover.map(reclassify_image_binary(4))
lcms_for = lcms_cover.map(reclassify_image_binary(1))

# Combine LCPRI and LCMS forests
conservative_forest = combine_collections_with_and(lcpri_for, lcms_for, 'conservativeForest') \
                        .map(add_year_property)

print('Conservative forest:', conservative_forest.aggregate_array('year').getInfo())

# Required forest year
required_forest_years = range(START_YEAR, END_YEAR)
actual_forest_years = conservative_forest.aggregate_array('year').distinct().sort().getInfo()

# Find missing forest years
missing_forest_years = set(required_forest_years) - (set(actual_forest_years))
print('Missing conservative forest years:', missing_forest_years)

# Determine available years in LCPRI and LCMS
lcpri_years = lcpri.aggregate_array('system:time_start') \
    .map(lambda d: ee.Date(d).get('year')).distinct().getInfo()
lcpri_years = set(lcpri_years)
lcms_years = lcms.aggregate_array('system:time_start') \
    .map(lambda d: ee.Date(d).get('year')).distinct().getInfo()
lcms_years = set(lcms_years)

available_from_lcpri = missing_forest_years.intersection(lcpri_years)
available_from_lcms = missing_forest_years.intersection(lcms_years)
available_from_either = available_from_lcpri.union(available_from_lcms)

if len(available_from_either) == 0:
    print("No missing forest years available from LCMS or LCPRI.") 
else:
    print("Missing forest years available from LCMS:", available_from_lcms)
    print("Missing forest years available from LCPRI:", available_from_lcpri)

def get_year_image(col, year):
    """Return the (binary) forest image for a given year from an ImageCollection."""
    return ee.Image(
        col.filter(ee.Filter.calendarRange(year, year, 'year')).first()
    )

final_conservative_forest = conservative_forest
missing_forest_years_sorted = sorted(missing_forest_years)

for y in missing_forest_years_sorted:
    src_img = None
    if y in lcpri_years:
        src_img = get_year_image(lcpri_for, y)
    elif y in lcms_years:
        src_img = get_year_image(lcms_for, y)
    
    if src_img is None:
        continue

    backfill_img = (
        src_img
        .rename('conservativeForest')
        .set('year', y)
    )

    final_conservative_forest = final_conservative_forest.merge(
        ee.ImageCollection([backfill_img])
    )

final_years = (
    final_conservative_forest
    .aggregate_array('year')
    .distinct()
    .sort()
    .getInfo()
)
print("Final conservative forest years:", final_years)

def rename_forest_band(img):
    year = ee.Number(img.get('year')).format('%d')
    band_name = ee.String('forest_').cat(year)
    
    img = img.rename(band_name)
    img = img.set('system:index', band_name)
    return img

final_conservative_forest_renamed = final_conservative_forest.map(rename_forest_band)

forest_by_year_raw = toBands_with_projection(final_conservative_forest_renamed)

forest_by_year_img = remove_to_bands_append2(
    forest_by_year_raw,
    og_ic=final_conservative_forest_renamed
)

print("Forest bands:", forest_by_year_img.bandNames().getInfo())

print(min(final_years))

Conservative forest: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021]
Missing conservative forest years: {1984, 2024, 2022, 2023}
Missing forest years available from LCMS: {2024, 2022, 2023}
Missing forest years available from LCPRI: set()
Final conservative forest years: [1985, 1986, 1987, 1988, 1989, 1990, 1991, 1992, 1993, 1994, 1995, 1996, 1997, 1998, 1999, 2000, 2001, 2002, 2003, 2004, 2005, 2006, 2007, 2008, 2009, 2010, 2011, 2012, 2013, 2014, 2015, 2016, 2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
Forest bands: ['forest_1985', 'forest_1986', 'forest_1987', 'forest_1988', 'forest_1989', 'forest_1990', 'forest_1991', 'forest_1992', 'forest_1993', 'forest_1994', 'forest_1995', 'forest_1996', 'forest_1997', 'forest_1998', 'forest_1999', 'forest_2000', 'forest_2001', 'forest_2002', 'forest_2003', 'forest_2004

In [21]:

assetID = f"{PROJECT_DIR}conservative_forest_{str(min(final_years))}_{str(max(final_years))}"
print("Exporting to asset ID:", assetID)

# Export to Asset
task = ee.batch.Export.image.toAsset(
    image=forest_by_year_img,
    description='Conservative forest',
    assetId=assetID,
    scale=30,
    crs=CRS,
    region=outputRegion.geometry(),  # geometry of study area
    maxPixels=1e13
)
task.start()

Exporting to asset ID: projects/ee-tymc5571-cbi-module/assets/conservative_forest_1985_2024
